# Steam Recommender Modeling\n
Baseline comparison and matrix factorization experiments.

In [ ]:
from pathlib import Path\n
import sys\n
\n
import pandas as pd\n
\n
ROOT = Path.cwd().parent\n
if str(ROOT) not in sys.path:\n
    sys.path.append(str(ROOT))\n
\n
from src.baselines import evaluate_recommendations, popularity_recommendations, random_recommendations\n
from src.data_loader import build_interaction_matrices, leave_one_out_split\n
from src.matrix_factorization import MatrixFactorizationSGD, build_test_items_by_user, evaluate_mf_leave_one_out

In [ ]:
DATA_DIR = ROOT / 'data'\n
interactions = pd.read_csv(DATA_DIR / 'interactions_filtered.csv')\n
train_df, test_df = leave_one_out_split(interactions)\n
\n
print('Train rows:', len(train_df))\n
print('Test rows:', len(test_df))

In [ ]:
train_data = build_interaction_matrices(train_df)\n
user_ids = sorted(test_df['user_id'].unique().tolist())\n
all_game_ids = sorted(train_df['app_id'].unique().tolist())\n
test_items_by_user_id = test_df.groupby('user_id')['app_id'].apply(list).to_dict()\n
\n
k_eval = 10\n
results = []\n
\n
pop_recs = popularity_recommendations(train_df, user_ids=user_ids, k=k_eval)\n
pop_scores = evaluate_recommendations(pop_recs, test_items_by_user_id, k=k_eval)\n
results.append({'model': 'Popularity', 'precision@10': pop_scores['precision_at_k'], 'recall@10': pop_scores['recall_at_k']})\n
\n
rand_recs = random_recommendations(train_df, user_ids=user_ids, all_game_ids=all_game_ids, k=k_eval, seed=42)\n
rand_scores = evaluate_recommendations(rand_recs, test_items_by_user_id, k=k_eval)\n
results.append({'model': 'Random', 'precision@10': rand_scores['precision_at_k'], 'recall@10': rand_scores['recall_at_k']})

In [ ]:
test_items_by_user_idx = build_test_items_by_user(test_df, train_data.user_to_idx, train_data.game_to_idx)\n\nfor latent_k in [20, 50, 100]:\n    model = MatrixFactorizationSGD(k=latent_k, reg=0.01, learning_rate=0.01, epochs=20, random_state=42)\n    model.fit(train_data.positive_matrix, verbose=True)\n    mf_scores = evaluate_mf_leave_one_out(model, train_data.binary_matrix, test_items_by_user_idx, k=k_eval)\n    results.append(\n        {\n            'model': f'MF (k={latent_k})',\n            'precision@10': mf_scores['precision_at_k'],\n            'recall@10': mf_scores['recall_at_k'],\n        }\n    )

In [ ]:
results_df = pd.DataFrame(results).sort_values('precision@10', ascending=False).reset_index(drop=True)\n
results_df